In [1]:
import happybase
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Row
from pyspark.sql import Window
import sys
from datetime import datetime
from tqdm import tqdm

spark = (
    SparkSession.builder
    .appName("Preprocess Batch")
    .master("spark://spark-master:7077")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

# Make sure the checkpoint table exists
# spark.sql("""
# CREATE TABLE IF NOT EXISTS cryptopredictions.batch_checkpoint (
#     table_name STRING,
#     last_processed_date DATE,
#     run_ts TIMESTAMP
# )
# STORED AS PARQUET;
# """)

# # Read checkpoint for your table
# checkpoint_df = (
#     spark.table("cryptopredictions.batch_checkpoint")
#     .filter(F.col("table_name") == "indexsnapshot")
# )

# # Get the latest checkpoint
# latest_checkpoint_row = checkpoint_df.orderBy(F.col("last_processed_date").desc()).limit(1).collect()

# if latest_checkpoint_row:
#     checkpoint = latest_checkpoint_row[0]["last_processed_date"]
#     print("Checkpoint:", checkpoint)
# else:
#     checkpoint = None  # No checkpoint yet
#     print("No checkpoint yet")
    
stock_df = spark.table("cryptopredictions.indexsnapshot")
# if checkpoint:
#     stock_df = stock_df.filter(F.col("PartitionDate") > checkpoint)
#     if stock_df.count() == 0:
#         print("No new data")
#         sys.exit(0)
        
# Sanity Check
bounds = (
    stock_df.select(
        F.min("Datetime").alias("first_datetime"),
        F.max("Datetime").alias("last_datetime")
    )
    .collect()[0]
)

print(f"[SANITY CHECK] Datetime range: {bounds.first_datetime} → {bounds.last_datetime}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/18 02:27:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/18 02:27:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/18 02:27:38 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/18 02:27:43 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
[Stage 1:======================================================>  (21 + 1) / 22]

[SANITY CHECK] Datetime range: 2025-11-15 11:40:35 → 2026-01-17 23:59:10


In [2]:
def aggregate_crypto(df, window_duration, granularity_label):
    return (
        df
        .groupBy(
            "IndexName",
            F.window("Datetime", window_duration).alias("w")
        )
        .agg(
            F.last("CurrentPrice").alias("close"),
        )
        .withColumn("granularity", F.lit(granularity_label))
        .withColumn("timestamp", F.col("w.start"))
        .drop("w")
    )
        
# Aggregate per granularity
stock_1m  = aggregate_crypto(stock_df, "1 minute", "1m")
stock_10m = aggregate_crypto(stock_df, "10 minutes", "10m")
stock_1d  = aggregate_crypto(stock_df, "1 day", "1d")

# Add SMA 7 & SMA 30 (ALL granularities)
def add_smas(df, periods=[7,30]):
    for period in periods:
        window_spec = Window.partitionBy("IndexName", "granularity").orderBy("timestamp").rowsBetween(-(period-1), 0)
        df = df.withColumn(f"SMA_{period}", F.avg("close").over(window_spec))
    return df

stock_1m  = add_smas(stock_1m)
stock_10m = add_smas(stock_10m)
stock_1d  = add_smas(stock_1d)

# Union all granularities
stock_agg = stock_1m.unionByName(stock_10m).unionByName(stock_1d)


[Stage 19:>                                                         (0 + 1) / 1]

+---------+-----+-----------+-------------------+-----+------+
|IndexName|close|granularity|          timestamp|SMA_7|SMA_30|
+---------+-----+-----------+-------------------+-----+------+
|     null| null|         1m|2025-11-18 12:04:00| null|  null|
|     null| null|         1m|2025-11-18 12:05:00| null|  null|
|     null| null|         1m|2025-11-18 12:06:00| null|  null|
|     null| null|         1m|2025-11-18 12:07:00| null|  null|
|     null| null|         1m|2025-11-18 12:15:00| null|  null|
+---------+-----+-----------+-------------------+-----+------+
only showing top 5 rows



In [6]:
def row_to_hbase(row):
    row_key = f"{row['IndexName']}#{row['timestamp']}#{row['granularity']}"
    return (
        row_key.encode(),
        {
            b"indicators:SMA_7": str(row['SMA_7']).encode(),
            b"indicators:SMA_30": str(row['SMA_30']).encode(),
        }
    )
      
# To HBase
run_ts = datetime.now()

connection = happybase.Connection(host='hbase')
table = connection.table('crypto_index_aggregates')
rows = stock_agg.collect()
for row in tqdm(rows):
    key, data = row_to_hbase(row)
    table.put(key, data)

print("HBase insertion and checkpoint update succeeded!")

end_ts = datetime.now()
print(f"Stock Hive -> Hbase ended at {end_ts}")
connection.close()
spark.stop()

100%|██████████| 282130/282130 [02:49<00:00, 1664.38it/s]                       


HBase insertion and checkpoint update succeeded!
Stock Hive -> Hbase ended at 2026-01-18 02:40:18.011232


In [1]:
import happybase
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
from tqdm import tqdm
from pyspark.sql import Window
from pyspark.sql import Row
spark = (
    SparkSession.builder
    .appName("hive -> hbase (agg)")
    .master("spark://spark-master:7077")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/15 20:59:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/15 20:59:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/15 20:59:46 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/15 20:59:46 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/01/15 20:59:46 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/01/15 20:59:46 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


In [ ]:
# Make sure the checkpoint table exists
spark.sql("""
CREATE TABLE IF NOT EXISTS cryptopredictions.batch_checkpoint (
    table_name STRING,
    last_processed_date DATE,
    run_ts TIMESTAMP
)
STORED AS PARQUET;
""")

25/12/17 22:40:29 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

In [ ]:
# Read checkpoint for your table
checkpoint_df = (
    spark.table("cryptopredictions.batch_checkpoint")
    .filter(F.col("table_name") == "indexsnapshot")
)

# Get the latest checkpoint
latest_checkpoint_row = checkpoint_df.orderBy(F.col("last_processed_date").desc()).limit(1).collect()
print("Checkpoint:", latest_checkpoint_row)

if latest_checkpoint_row:
    checkpoint = latest_checkpoint_row[0]["last_processed_date"]
else:
    checkpoint = None  # No checkpoint yet

crypto_df = spark.table("cryptopredictions.indexsnapshot")
if checkpoint:
    crypto_df = crypto_df.filter(F.col("PartitionDate") > checkpoint)
    if crypto_df.count() == 0:
        print("No new data")
else:
    print("No checkpoint data")

Checkpoint: [Row(table_name='indexsnapshot', last_processed_date=datetime.date(2025, 12, 15), run_ts=datetime.datetime(2025, 12, 17, 23, 52, 35, 621218))]


In [6]:
crypto_df.show(1,vertical=True)

-RECORD 0-----------------------------------------
 IndexName                  | SNP                 
 Datetime                   | 2025-11-19 15:00:47 
 CurrentPrice               | 6671.7099609375     
 CurrentVolume              | 362266710           
 OpeningPrice               | 6625.83984375       
 LowestDayPrice             | 6618.47998046875    
 HighestDayPrice            | 6675.14990234375    
 LowestYearlyPrice          | 4835.0400390625     
 HighestYearlyPrice         | 6920.33984375       
 FiftyDayAveragePrice       | 6709.807392578125   
 TwoHundredDaysAveragePrice | 6154.74173828125    
 TenDayAverageVolume        | 5444071000          
 ThreeMonthAverageVolume    | 5376607538          
 YearOverYearPriceChange    | 0.11833648134246547 
 PartitionDate              | 2025-11-19          
only showing top 1 row



In [7]:
crypto_df.filter(F.col("IndexName")=="SNP").filter(F.col("Datetime")>"2025-12-01 15:00:00").orderBy("Datetime", ascending = True).select(["Datetime","CurrentPrice", "CurrentVolume"]).show(100)

+-------------------+----------------+-------------+
|           Datetime|    CurrentPrice|CurrentVolume|
+-------------------+----------------+-------------+
|2025-12-01 15:00:10| 6830.0400390625|    330457165|
|2025-12-01 15:01:11|6831.02978515625|    337219596|
|2025-12-01 15:02:11|6828.68994140625|    343917583|
|2025-12-01 15:03:11|6823.56005859375|    351805321|
|2025-12-01 15:04:10|   6822.91015625|    358189371|
|2025-12-01 15:05:11|  6821.169921875|    365138791|
|2025-12-01 15:06:11|6824.52978515625|    372103145|
|2025-12-01 15:07:10|6825.56982421875|    378527612|
|2025-12-01 15:08:11|6824.81005859375|    385164401|
|2025-12-01 15:09:10|6826.89013671875|    391747195|
|2025-12-01 15:10:11|6821.81982421875|    398979255|
|2025-12-01 15:11:11|  6824.080078125|    405023861|
|2025-12-01 15:12:10|   6821.91015625|    411686311|
|2025-12-01 15:13:11| 6824.7900390625|    417545299|
|2025-12-01 15:14:11|6824.77978515625|    423398420|
|2025-12-01 15:15:10| 6820.2900390625|    4292

In [8]:
def aggregate_crypto(df, window_duration, granularity_label):
    if granularity_label != "1d":
        return (
            df
            .groupBy(
                "IndexName",
                F.window("Datetime", window_duration).alias("w")
            )
            .agg(
                F.first("CurrentPrice").alias("open"),
                F.max("HighestDayPrice").alias("high"),
                F.min("LowestDayPrice").alias("low"),
                F.last("CurrentPrice").alias("close"),
                F.last("CurrentVolume").alias("max_volume")
            )
            .withColumn("granularity", F.lit(granularity_label))
            .withColumn("timestamp", F.col("w.start"))
            .drop("w")
        )
    elif granularity_label == "1d":
        return (
                df
                .groupBy(
                    "IndexName",
                    F.window("Datetime", window_duration).alias("w")
                )
                .agg(
                    F.first("CurrentPrice").alias("open"),
                    F.max("HighestDayPrice").alias("high"),
                    F.min("LowestDayPrice").alias("low"),
                    F.last("CurrentPrice").alias("close"),
                    F.last("CurrentVolume").alias("max_volume"),
                    # Indicators -> created at around 4:30 -> that's why we take last
                    F.last("FiftyDayAveragePrice").alias("ma_50_price"),
                    F.last("TwoHundredDaysAveragePrice").alias("ma_200_price"),
                    F.last("TenDayAverageVolume").alias("ma_10_volume"),
                    F.last("ThreeMonthAverageVolume").alias("ma_3m_volume"),
                    F.last("YearOverYearPriceChange").alias("yoy_price_change"),
                )
                .withColumn("granularity", F.lit(granularity_label))
                .withColumn("timestamp", F.col("w.start"))
                .drop("w")
            )

crypto_1m  = aggregate_crypto(crypto_df, "1 minute", "1m")
crypto_10m = aggregate_crypto(crypto_df, "10 minutes", "10m")
crypto_1d  = aggregate_crypto(crypto_df, "1 day", "1d")


def add_daily_indicator_nulls(df):
    return (
        df
        .withColumn("ma_50_price", F.lit(None).cast(T.DoubleType()))
        .withColumn("ma_200_price", F.lit(None).cast(T.DoubleType()))
        .withColumn("ma_10_volume", F.lit(None).cast(T.LongType()))
        .withColumn("ma_3m_volume", F.lit(None).cast(T.LongType()))
        .withColumn("yoy_price_change", F.lit(None).cast(T.DoubleType()))
    )
crypto_1m  = add_daily_indicator_nulls(crypto_1m)
crypto_10m = add_daily_indicator_nulls(crypto_10m)

crypto_agg = crypto_1m.unionByName(crypto_10m).unionByName(crypto_1d)

crypto_agg.filter(F.col("granularity") == "1m").show(5)
crypto_agg.filter(F.col("granularity") == "10m").show(5)
crypto_agg.filter(F.col("granularity") == "1d").show(5)

+---------+---------------+--------------+----------------+---------------+----------+-----------+-------------------+-----------+------------+------------+------------+----------------+
|IndexName|           open|          high|             low|          close|max_volume|granularity|          timestamp|ma_50_price|ma_200_price|ma_10_volume|ma_3m_volume|yoy_price_change|
+---------+---------------+--------------+----------------+---------------+----------+-----------+-------------------+-----------+------------+------------+------------+----------------+
|      NIM|  22758.2734375|22778.28515625|  22446.88671875|  22758.2734375|1782457000|         1m|2025-11-19 15:09:00|       null|        null|        null|        null|            null|
|      SNP| 6627.919921875|       6689.75|6618.47998046875| 6627.919921875|1152850000|         1m|2025-11-19 17:07:00|       null|        null|        null|        null|            null|
|      DJI|   45991.640625|46299.12890625|  45963.41015625|   459

+---------+----------------+--------------+---------------+---------------+----------+-----------+-------------------+-----------+------------+------------+------------+----------------+
|IndexName|            open|          high|            low|          close|max_volume|granularity|          timestamp|ma_50_price|ma_200_price|ma_10_volume|ma_3m_volume|yoy_price_change|
+---------+----------------+--------------+---------------+---------------+----------+-----------+-------------------+-----------+------------+------------+------------+----------------+
|      SNP|6708.43017578125|        6754.5| 6695.259765625| 6701.919921875|1339686000|        10m|2025-11-17 18:00:00|       null|        null|        null|        null|            null|
|      NIM| 23365.689453125| 23365.7890625|23250.509765625|23365.689453125|4541070000|        10m|2025-11-29 11:10:00|       null|        null|        null|        null|            null|
|      NIM| 23539.583984375|   23680.03125|        23506.0|23537.

+---------+----------------+--------------+----------------+----------------+----------+-----------+-------------------+-----------------+------------------+------------+------------+-------------------+
|IndexName|            open|          high|             low|           close|max_volume|granularity|          timestamp|      ma_50_price|      ma_200_price|ma_10_volume|ma_3m_volume|   yoy_price_change|
+---------+----------------+--------------+----------------+----------------+----------+-----------+-------------------+-----------------+------------------+------------+------------+-------------------+
|      SNP| 6671.7099609375|       6689.75|6574.31982421875|   6642.16015625|3101765000|         1d|2025-11-19 00:00:00|6709.807392578125|  6154.74173828125|  5444071000|  5376607538|0.11833648134246547|
|      DJI|  47954.98828125| 48133.5390625|  47871.51171875|  47954.98828125| 456100000|         1d|2025-12-06 00:00:00|      46900.94875|  44028.9096484375|   534879000|   519821718|0

In [9]:
crypto_1d.orderBy(['IndexName', 'timestamp'], ascending = False).select(['timestamp', 'IndexName', 'yoy_price_change']).show(10)

+-------------------+---------+-------------------+
|          timestamp|IndexName|   yoy_price_change|
+-------------------+---------+-------------------+
|2025-12-15 00:00:00|      SNP|0.12402373173149613|
|2025-12-14 00:00:00|      SNP|0.12402373173149613|
|2025-12-13 00:00:00|      SNP|0.12402373173149613|
|2025-12-12 00:00:00|      SNP| 0.1404557159447646|
|2025-12-11 00:00:00|      SNP|0.13805910775149763|
|2025-12-10 00:00:00|      SNP|0.12430904220651935|
|2025-12-09 00:00:00|      SNP|0.13448412459537185|
|2025-12-08 00:00:00|      SNP|0.13506856959898395|
|2025-12-07 00:00:00|      SNP|0.13506856959898395|
|2025-12-06 00:00:00|      SNP|0.13506856959898395|
+-------------------+---------+-------------------+
only showing top 10 rows



In [16]:
max_date = crypto_df.agg(F.max("PartitionDate")).collect()[0][0]
print(max_date)

[Stage 21:==================================================>       (7 + 1) / 8]

2025-12-15


In [10]:
def row_to_hbase(row):
    row_key = f"{row['IndexName']}#{row['timestamp']}#{row['granularity']}"
    if row['granularity'] == "1d":
        return (
        row_key.encode(),
        {
            b"ohlc:open": str(row['open']).encode(),
            b"ohlc:high": str(row['high']).encode(),
            b"ohlc:low": str(row['low']).encode(),
            b"ohlc:close": str(row['close']).encode(),
            b"indicators:ma_50_price": str(row['ma_50_price']).encode(),
            b"indicators:ma_200_price": str(row['ma_200_price']).encode(),
            b"indicators:ma_10_volume": str(row['ma_10_volume']).encode(),
            b"indicators:yoy_price_change": str(row['yoy_price_change']).encode(),
        }
    )
    else:
        return (
            row_key.encode(),
            {
                b"ohlc:open": str(row['open']).encode(),
                b"ohlc:high": str(row['high']).encode(),
                b"ohlc:low": str(row['low']).encode(),
                b"ohlc:close": str(row['close']).encode(),
            }
        )
      
# To HBase
run_ts = datetime.now()

connection = happybase.Connection(host='hbase')
table = connection.table('crypto_index_aggregates')
rows = crypto_agg.collect()
for row in tqdm(rows):
    key, data = row_to_hbase(row)
    table.put(key, data)

max_date = crypto_df.agg(F.max("PartitionDate")).collect()[0][0]

new_checkpoint = spark.createDataFrame([Row(
    table_name="indexsnapshot",
    last_processed_date=max_date,
    run_ts=run_ts
)])

(new_checkpoint.write
    .mode("append")
    .format("hive")
    .saveAsTable("cryptopredictions.batch_checkpoint"))

print("HBase insertion and checkpoint update succeeded!")

100%|██████████| 126923/126923 [01:32<00:00, 1373.86it/s]                       
26/01/15 21:04:26 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


HBase insertion and checkpoint update succeeded!


In [11]:
spark.stop()